# Data Cleaning Pipeline

Remove constant columns (zero variance) and save as lightweight `.parquet` files.

In [ ]:
import pandas as pd
import numpy as np
import gc

# --- 1. Optimized loading with minimal dtypes ---
def load_smart_types(filepath):
    print(f"Loading {filepath}...")
    cols = pd.read_csv(filepath, nrows=0).columns
    dtype_map = {}
    for col in cols:
        if col in ['wip', 'investissement', 'satisfaction']:
            dtype_map[col] = 'float32'
        elif col == 'id':
            dtype_map[col] = 'string'
        else:
            dtype_map[col] = 'int8'
    return pd.read_csv(filepath, dtype=dtype_map, engine='c')

# --- 2. Process training set ---
df_train = load_smart_types('data/train.csv')

print("Searching for constant columns (variance = 0)...")
targets = ['wip', 'investissement', 'satisfaction']
features = [c for c in df_train.columns if c not in targets and c != 'id']

const_cols = [col for col in features if df_train[col].min() == df_train[col].max()]
print(f"Found {len(const_cols)} constant columns to remove.")

df_train.drop(columns=const_cols, inplace=True)
print(f"Train reduced to {df_train.shape[1]} columns.")

print("Saving 'train_cleaned.parquet'...")
df_train.to_parquet('data/train_cleaned.parquet', index=False, compression='zstd')

del df_train
gc.collect()

# --- 3. Process test set (same columns) ---
df_test = load_smart_types('data/test.csv')
cols_to_drop_test = [c for c in const_cols if c in df_test.columns]
df_test.drop(columns=cols_to_drop_test, inplace=True)
print(f"Test reduced to {df_test.shape[1]} columns.")

print("Saving 'test_cleaned.parquet'...")
df_test.to_parquet('data/test_cleaned.parquet', index=False, compression='zstd')

print("\nDone! Use the '_cleaned.parquet' files for next steps.")

# Remove Highly Correlated Features (>98% Pearson Correlation)

In [ ]:
import pandas as pd
import numpy as np
import gc
import os

THRESHOLD = 0.98

# 1. Load cleaned train
print("Loading 'data/train_cleaned.parquet'...")
df_train = pd.read_parquet('data/train_cleaned.parquet')

targets = ['wip', 'investissement', 'satisfaction']
features = [c for c in df_train.columns if c not in targets and c != 'id']

# 2. Compute correlation on a sample (10k rows is sufficient)
corr_file = 'data/corr_matrix_sample.parquet'
if os.path.exists(corr_file):
    print("Loading cached correlation matrix...")
    corr_matrix = pd.read_parquet(corr_file)
else:
    print("Computing correlation on 10,000-row sample...")
    sample_df = df_train[features].sample(n=10000, random_state=42)
    corr_matrix = sample_df.corr().abs()
    corr_matrix.to_parquet(corr_file)
    del sample_df
    gc.collect()

# 3. Filter correlated features
print(f"Filtering with threshold {THRESHOLD}...")
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > THRESHOLD)]
print(f"Found {len(to_drop)} correlated features to remove.")

# 4. Drop from full dataset
if len(to_drop) > 0:
    df_train.drop(columns=to_drop, inplace=True)

print(f"Train final shape: {df_train.shape}")
df_train.to_parquet(f'data/train_uncorrelated_{int(THRESHOLD*100)}.parquet', index=False, compression='zstd')

del df_train, corr_matrix, upper
gc.collect()

# 5. Apply same filtering to test
print("Applying to test set...")
df_test = pd.read_parquet('data/test_cleaned.parquet')
cols_to_drop_test = [c for c in to_drop if c in df_test.columns]
if cols_to_drop_test:
    df_test.drop(columns=cols_to_drop_test, inplace=True)

df_test.to_parquet(f'data/test_uncorrelated_{int(THRESHOLD*100)}.parquet', index=False, compression='zstd')
print("Done!")

Chargement de 'data/train_uncorrelated.parquet'...


Calcul de la corrélation sur un échantillon de 10 000 lignes...
Sauvegarde de la matrice dans data/corr_matrix_sample.parquet pour usage futur...
Filtrage avec seuil {THRESHOLD}...
Trouvé 154 features corrélées (sur l'échantillon).
Suppression dans le TRAIN complet...
Train final shape: (76905, 2372)
Application au TEST...
Terminé !


# Feature Importance Analysis (LightGBM)

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
import gc

# 1. Load cleaned dataset
print("Loading 'data/train_uncorrelated.parquet'...")
df_train = pd.read_parquet('data/train_uncorrelated.parquet')
print(f"Shape: {df_train.shape}")

# 2. Configuration
params = {
    'objective': 'regression',
    'metric': 'mae',
    'n_estimators': 7000,
    'learning_rate': 0.04,
    'num_leaves': 31,
    'subsample': 0.8,
    'colsample_bytree': 0.6,
    'n_jobs': -1,
    'verbosity': -1,
}

# 3. Chained regression with feature importance capture
targets = ['wip', 'investissement', 'satisfaction']
features_base = [c for c in df_train.columns if c not in targets and c != 'id']

y_wip = df_train['wip']
y_inv = df_train['investissement']
y_sat = df_train['satisfaction']

print(f"Starting chained regression on {len(features_base)} features...")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
scores = []
feature_importance_list = []

for fold, (train_idx, val_idx) in enumerate(kf.split(df_train)):
    print(f"\n--- Fold {fold+1}/5 ---")
    
    X_tr = df_train.iloc[train_idx][features_base]
    X_val = df_train.iloc[val_idx][features_base]
    
    # Step 1: WIP
    dtrain = lgb.Dataset(X_tr, label=y_wip.iloc[train_idx])
    dval = lgb.Dataset(X_val, label=y_wip.iloc[val_idx], reference=dtrain)
    model_wip = lgb.train(params, dtrain, valid_sets=[dval], 
                          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    wip_pred_tr = model_wip.predict(X_tr)
    wip_pred_val = model_wip.predict(X_val)
    del dtrain, dval, model_wip
    gc.collect()

    # Step 2: Investment
    dtrain = lgb.Dataset(X_tr, label=y_inv.iloc[train_idx])
    dval = lgb.Dataset(X_val, label=y_inv.iloc[val_idx], reference=dtrain)
    model_inv = lgb.train(params, dtrain, valid_sets=[dval], 
                          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    inv_pred_tr = model_inv.predict(X_tr)
    inv_pred_val = model_inv.predict(X_val)
    del dtrain, dval, model_inv
    gc.collect()
    
    # Step 3: Satisfaction (with chained predictions)
    X_tr_aug = X_tr.copy()
    X_val_aug = X_val.copy()
    X_tr_aug['pred_wip'] = wip_pred_tr
    X_tr_aug['pred_inv'] = inv_pred_tr
    X_val_aug['pred_wip'] = wip_pred_val
    X_val_aug['pred_inv'] = inv_pred_val
    
    dtrain = lgb.Dataset(X_tr_aug, label=y_sat.iloc[train_idx])
    dval = lgb.Dataset(X_val_aug, label=y_sat.iloc[val_idx], reference=dtrain)
    model_sat = lgb.train(params, dtrain, valid_sets=[dval], 
                          callbacks=[lgb.early_stopping(50), lgb.log_evaluation(500)])
    
    # Capture feature importance
    fold_imp = pd.DataFrame({
        'feature': model_sat.feature_name(),
        'gain': model_sat.feature_importance(importance_type='gain'),
        'fold': fold + 1
    })
    feature_importance_list.append(fold_imp)

    val_preds = model_sat.predict(X_val_aug)
    score = np.mean(np.abs(y_sat.iloc[val_idx] - val_preds) < 0.05)
    scores.append(score)
    print(f"Fold {fold+1} Accuracy: {score:.4%}")
    
    del X_tr, X_val, X_tr_aug, X_val_aug, dtrain, dval, model_sat
    gc.collect()

print(f"\nFINAL CHAINED SCORE: {np.mean(scores):.4%}")

Chargement de 'data/train_uncorrelated.parquet'...
Dimensions : (76905, 2526)
Démarrage de la CHAIN REGRESSION sur 2522 features...

--- Fold 1/5 ---
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[7000]	valid_0's l1: 1.21038e+06
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[7000]	valid_0's l1: 664.725
Training until validation scores don't improve for 50 rounds
[500]	valid_0's l1: 0.0418845
[1000]	valid_0's l1: 0.039155
[1500]	valid_0's l1: 0.0376507
[2000]	valid_0's l1: 0.0366735
[2500]	valid_0's l1: 0.0359619
[3000]	valid_0's l1: 0.0354364
[3500]	valid_0's l1: 0.0351254
[4000]	valid_0's l1: 0.034928
[4500]	valid_0's l1: 0.0346807
[5000]	valid_0's l1: 0.0345546
[5500]	valid_0's l1: 0.0344439
[6000]	valid_0's l1: 0.0343361
[6500]	valid_0's l1: 0.0341872
[7000]	valid_0's l1: 0.0340949
Did not meet early stopping. Best iteration is:
[7000]	valid_0's l1: 0.0340949

In [ ]:
# Feature importance analysis across folds

importances = pd.concat(feature_importance_list)
avg_importance = importances.groupby('feature')['gain'].mean().sort_values(ascending=False).reset_index()

# Filter out artificial chained features
features_ranked = avg_importance[~avg_importance['feature'].isin(['pred_wip', 'pred_inv'])]

print("Top 10 most important features:")
print(features_ranked.head(10))

# Save full ranking
filename = 'data/feature_importance_ranking.csv'
features_ranked.to_csv(filename, index=False)
print(f"\nFull ranking saved to: {filename}")

 Top 10 Features les plus puissantes :
            feature          gain
0    param_year0_f5  19598.462927
2   param_year0_f14   2888.041550
3   param_year1_f14   1646.927545
4   param_year0_f15    645.713060
5   param_year1_f15    597.116350
6   param_year2_f14    476.624709
7      demand_492_0    247.847381
8   param_year0_f16    215.524939
9   param_year2_f15    207.452181
11     demand_495_4    145.702861

Liste 'top_features_list' créée avec les 600 meilleures features.
Tu peux maintenant utiliser cette liste dans ta recherche Optuna !
Classement complet sauvegardé dans : data/feature_importance_ranking.csv


# Hyperparameter Search with Optuna

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import KFold
import gc

# 1. Load feature importance ranking and select top features
print("1. Loading feature importance ranking...")
df_importance = pd.read_csv('data/feature_importance_ranking.csv')

TOP_N = 500
top_features = df_importance.head(TOP_N)['feature'].tolist()
print(f"Selected top {TOP_N} features.")

# 2. Load and reduce dataset
print("2. Loading training data...")
df_train = pd.read_parquet('data/train_uncorrelated.parquet')

targets = ['wip', 'investissement', 'satisfaction']
df_train = df_train[top_features + targets]
print(f"Reduced dataset shape: {df_train.shape}")

y_wip = df_train['wip']
y_inv = df_train['investissement']
y_sat = df_train['satisfaction']

# 3. Optuna objective function
def objective(trial):
    param_grid = {
        'objective': 'regression',
        'metric': 'mae',
        'verbosity': -1,
        'n_jobs': -1,
        'n_estimators': 8000,
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 128),
        'subsample': trial.suggest_float('subsample', 0.5, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 100),
    }
    
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in kf.split(df_train):
        X_tr = df_train.iloc[train_idx][top_features]
        X_val = df_train.iloc[val_idx][top_features]
        
        # Step 1: WIP
        dtrain_wip = lgb.Dataset(X_tr, label=y_wip.iloc[train_idx])
        model_wip = lgb.train(param_grid, dtrain_wip)
        pred_wip_tr = model_wip.predict(X_tr)
        pred_wip_val = model_wip.predict(X_val)
        
        # Step 2: Investment
        dtrain_inv = lgb.Dataset(X_tr, label=y_inv.iloc[train_idx])
        model_inv = lgb.train(param_grid, dtrain_inv)
        pred_inv_tr = model_inv.predict(X_tr)
        pred_inv_val = model_inv.predict(X_val)
        
        # Step 3: Satisfaction (chained)
        X_tr_aug = X_tr.copy()
        X_val_aug = X_val.copy()
        X_tr_aug['pred_wip'] = pred_wip_tr
        X_tr_aug['pred_inv'] = pred_inv_tr
        X_val_aug['pred_wip'] = pred_wip_val
        X_val_aug['pred_inv'] = pred_inv_val
        
        dtrain_sat = lgb.Dataset(X_tr_aug, label=y_sat.iloc[train_idx])
        dval_sat = lgb.Dataset(X_val_aug, label=y_sat.iloc[val_idx], reference=dtrain_sat)
        model_sat = lgb.train(param_grid, dtrain_sat, 
                              valid_sets=[dval_sat],
                              callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(0)])
        
        val_preds = model_sat.predict(X_val_aug)
        scores.append(np.mean(np.abs(y_sat.iloc[val_idx] - val_preds) < 0.05))
        
        del dtrain_wip, dtrain_inv, dtrain_sat, model_wip, model_inv, model_sat
        del X_tr, X_val, X_tr_aug, X_val_aug
        gc.collect()
    
    return np.mean(scores)

# 4. Run optimization
print("Starting Optuna optimization...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)

print(f"\nBest hyperparameters: {study.best_params}")
print(f"Best CV score (3-fold): {study.best_value:.4%}")

import json
with open('best_params_optuna.json', 'w') as f:
    json.dump(study.best_params, f)
print("Parameters saved to 'best_params_optuna.json'")

c:\Users\romai\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Chargement du classement des features...
Top 500 features sélectionnées.
2. Chargement des données d'entraînement...


[I 2025-12-16 14:12:06,618] A new study created in memory with name: no-name-a66d2a79-f40c-4201-8c12-13e73f9a4d40


Dimensions réduites pour Optuna : (76905, 503)
Démarrage de l'optimisation Optuna...


[I 2025-12-16 14:19:55,270] Trial 0 finished with value: 0.7054027696508679 and parameters: {'learning_rate': 0.01967921704859526, 'num_leaves': 39, 'subsample': 0.8170250155877481, 'colsample_bytree': 0.8791827086320692, 'reg_alpha': 6.084665990687893, 'reg_lambda': 8.947183963714485, 'min_child_samples': 52}. Best is trial 0 with value: 0.7054027696508679.
[I 2025-12-16 14:27:47,191] Trial 1 finished with value: 0.70013653208504 and parameters: {'learning_rate': 0.04780941340983147, 'num_leaves': 55, 'subsample': 0.9051624643326849, 'colsample_bytree': 0.8183161914107251, 'reg_alpha': 6.311859555848111, 'reg_lambda': 1.833998572210751, 'min_child_samples': 57}. Best is trial 0 with value: 0.7054027696508679.
[I 2025-12-16 14:37:49,030] Trial 2 finished with value: 0.6913724725310448 and parameters: {'learning_rate': 0.07257469671340147, 'num_leaves': 112, 'subsample': 0.6742665527078845, 'colsample_bytree': 0.7330761004869528, 'reg_alpha': 6.568703279985471, 'reg_lambda': 0.131367389

KeyboardInterrupt: 